## Load spiking output of a circuit simulation campaign

Notebook for downloading the spiking output of a successfully run simulation campaign. Instead of downloading the entire circuit along with it, only its connectivity matrix is downloaded and loaded using Connectome-Utilities.

### Use
Simply following the instructions

In [ ]:
import pandas

from obi_auth import get_token
from entitysdk.client import Client
from entitysdk.models import SimulationCampaign
from obi_auth import get_token
from obi_notebook.get_projects import get_projects
from obi_notebook.get_environment import get_environment

from matplotlib import pyplot as plt

## Get authentication token and choose a project context

In [ ]:
token = get_token(environment=get_environment(), auth_mode="daf")
project_context = get_projects(token=token)

## Initialize the client with the token and project context

In [ ]:
client = Client(environment=get_environment(), project_context=project_context, token_manager=token)

## Select SimulationCampaigns
Either by:
1. Specifying SimulationCampaign IDs
2. Selecting from the widget

In [ ]:
from obi_notebook import get_entities

# Optional: Download using unique ID
entity_ID = "<SIMULATION-CAMPAIGN-IDs>"  # <<< FILL IN A LIST OF UNIQUE SIMULATION CAMPAIGN IDENTIFIERS HERE


if entity_ID != "<SIMULATION-CAMPAIGN-IDs>":
    simulation_campaign_ids = [entity_ID]
else:
# Alternative: Select from a table of entities
    simulation_campaign_ids = []
    simulation_campaign_ids = get_entities.get_entities("simulation-campaign", token, simulation_campaign_ids,
                                          project_context=project_context,
                                          multi_select=False,
                                          page_size=100)

## Print the selected IDs
If you used the widget you might want to copy these the entity_IDs parameter so they are already selected for next time

In [ ]:
print(simulation_campaign_ids)

# Stage the spike output of the simulation campaign - and its circuit connectivity - locally

In [ ]:
from campaign_loading import download_circuit_connectivity
from campaign_loading import stage_sim_campaign_spikes

simulation_campaign_id = simulation_campaign_ids[0]
output_dir = f"./campaigns/{simulation_campaign_id}"

simulation_campaign = client.get_entity(entity_type=SimulationCampaign, entity_id=simulation_campaign_id)

display(f"Downloading connectivity to {output_dir}/connectivity")
matrix_file_dict = download_circuit_connectivity(client, simulation_campaign=simulation_campaign, output_dir=output_dir)
display(f"Downloading campaign spikes to {output_dir}")
campaign_spikes = stage_sim_campaign_spikes(client=client, simulation_campaign_id=simulation_campaign_id,
                                     output_dir=output_dir)

# Load circuit connectivity matrix
We load the circuit connectivity using the `Connectome-Utilities` package [(Link)](https://github.com/openbraininstitute/ConnectomeUtilities).

A widget will be displayed that allows you to select a circuit connectivity matrix to load. Select one and press the `Load` button.

In [ ]:
from ipywidgets import widgets
from conntility import ConnectivityMatrix

sel_matrix = widgets.Dropdown(
    description="Select matrix",
    options=matrix_file_dict
)
loader_button = widgets.Button(description="Load")

connectivity = []

def loading_function(event):
    connectivity.clear()
    connectivity.append(ConnectivityMatrix.from_h5(sel_matrix.value))
loader_button.on_click(loading_function)

display(widgets.VBox([sel_matrix, loader_button]))

## Working with the connectivity matrix

Under [this link](https://github.com/openbraininstitute/ConnectomeUtilities/blob/main/README.md) you can find the documentation of the Connectome-Utilities package.

Briefly: You can access the adjacency matrix as a `scipy.sparse` matrix, where the value of nonzero entries denotes the number of synaptic connections from one neuron to another.

In [ ]:
display(connectivity[0].matrix)

You can also access information about the neurons, such as their types and locations

In [ ]:
display(connectivity[0].vertices)

### Select a population of interest

We can use the `.index` function to select a sub-population based on neuron properties.

In the example below, we select excitatory neurons (value of `synapse_class` equal to `EXC`).

Feel free to play around with the functionality and change the filters applied.

Finally, `submatrix` represents the connectivity in that subpopulation only.

In [ ]:
submatrix = connectivity[0].index("synapse_class").eq("EXC")

# Uncomment the following line to keep the entire population and load all spikes.
# submatrix = connectivity[0]

display(submatrix.matrix)

### Example: Connectivity strengths in pathways
As an example, we can calculate the number of synapses between populations of different neuron morphology types (`mtype`).

In [ ]:
display(submatrix.condense("mtype").array)

## Load spikes of simulations

We use `libsonata` to load spikes from the downloaded .h5 files.

The widget will display the downloaded spike files, one for each simulation in the campaign. Select one and press the load button.

### Note
We will only load spikes for the neurons in `submatrix` (see filtering applied above).

In [ ]:
from libsonata import SpikeReader

sel_spike_file = widgets.Dropdown(
    description="Simulation",
    options=campaign_spikes
)
spike_loader_button = widgets.Button(description="Load")

class Spikes:
    df = None
spikes = Spikes()

def spike_loading_function(event):
    rdr = SpikeReader(sel_spike_file.value)
    spike_population = rdr[rdr.get_population_names()[0]]
    spikes.df = pandas.DataFrame(
        spike_population.get(node_ids=list(submatrix.gids)),
        columns=["node_ids", "time"]
    )
    
spike_loader_button.on_click(spike_loading_function)

display(widgets.VBox([sel_spike_file, spike_loader_button]))

Spikes are represented by the `node_id` of the spiking neuron and the time of the spike in ms.

You can cross-reference `node_id` with the same column in `submatrix.vertices` to understand which neuron is spiking.

In [ ]:
display(spikes.df)

### Example: Spike scatter plot

In [ ]:
plt.scatter(spikes.df["time"], spikes.df["node_ids"], s=1)
plt.gca().set_xlabel("Time (ms)")
plt.gca().set_ylabel("Neuron id")

### Example: Firing rate PSTH

In [ ]:
import numpy

dt = 50
bins = numpy.arange(0, spikes.df["time"].max() + dt, dt)
n = len(submatrix)

rate = (1000 / dt) * numpy.histogram(spikes.df["time"], bins=bins)[0] / n

plt.plot(bins[:-1], rate)
plt.gca().set_xlabel("Time (ms)")
plt.gca().set_ylabel("Mean rate (hz)")